In [1]:
!pip uninstall -y bitsandbytes triton torch torchvision torchaudio
!pip install torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install bitsandbytes==0.45.0
!pip install transformers accelerate peft

import torch, bitsandbytes as bnb
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)


Found existing installation: triton 3.5.0
Uninstalling triton-3.5.0:
  Successfully uninstalled triton-3.5.0
Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 137.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 107.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━

In [2]:
from huggingface_hub import login
login()

mapping = {}

mapping["periodontal_abscess"] = {
    "false toothache",
    "gum pain",
    "pain when touched",
    "sensitivity when biting",
    "unusual taste",
    "salty-tasting fluid",
    "gum swelling with pus",
    "pain on palpation",
}

mapping["simple_cavities"] = {
    "short sharp pain",
    "pain to cold",
    "pain to sweet",
    "pain to sour",
    "discomfort when brushing",
    "pain to stimuli",
}

mapping["acute_apical_periodontitis"] = {
    "pain when chewing",
    "pain when touched",
    "tooth feels higher",
    "mobile tooth",
    "discomfort on palpation",
    "pain on percussion",
}

mapping["chronic_apical_periodontitis"] = {
    "gum swelling",
    "salty-tasting fluid",
    "pressure when biting",
    "dull ache in the tooth",
    "sensitivity to percussion",
    "sensitivity to palpation",
}

mapping["pericoronitis"] = {
    "continuous pain in the wisdom tooth area",
    "pain when chewing",
    "pain when swallowing",
    "cannot open mouth fully",
    "swelling over the wisdom tooth",
    "inflamed gum",
    "partially erupted wisdom tooth",
    "salty-tasting fluid",
}

mapping["reversible_pulpitis"] = {
    "short sharp pain",
    "pain to cold",
    "pain to sweet",
}

mapping["irreversible_pulpitis"] = {
    "spontaneous pain",
    "strong prolonged pain",
    "pain to cold",
    "pain to heat",
    "throbbing pain radiating to the ear",
    "slight pain on percussion",
    "tolerable sensitivity on palpation",
}

mapping["pulp_necrosis"] = {
    "spontaneous pain",
    "intense short pain",
    "pain to heat",
    "pressure when biting",
    "sensitivity to percussion",
    "sensitivity to palpation",
}

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

lora_dir = r"/content/drive/My Drive/pacient-llama31-8b-lora-FINAL/checkpoint-52"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map={"": 0} if device == "cuda" else None,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

base_model.config.use_cache = False

model = PeftModel.from_pretrained(
    base_model,
    lora_dir,
)

model.eval()
print("Model + LoRA încărcate!")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model + LoRA încărcate!


In [5]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Set
import random
import torch

SYSTEM_PROMPT_TEMPLATE = """
You are NOT a language model in this exercise.
You are playing the role of a REAL HUMAN PATIENT in a medical simulation scenario.

Your role:
- You are a patient talking to a dental student.
- Your goal is to help the student practice identifying symptoms.
- You do not know diagnoses, analyze causes, explain medicine. You just tell how you feel.

INSIDE INFORMATION (only for you, do not reveal it):
- Real diagnosis (hidden): {diagnosis}
- Complete and final list of your real symptoms:
{symptom_bullets}

ABSOLUTE RULES (take precedence over any other instructions):
1. The symptoms in the list above represent your REALITY.
- If the student asks you about a symptom FROM THE LIST → answer YES, you have it.
- If they ask about something that is NOT on the list → you answer NO, you do not have that symptom.
(This rule is mandatory and cannot be broken.)

2. Never invent new symptoms.
3. Do not add unsolicited symptoms unless the student explicitly asks
“Do you have any other problems?” or something similar.
4. Do not use the labels on the list robotically.
Transform the symptoms into natural, realistic sentences.
5. Answer only the student’s question, in 1–3 short sentences.
6. Never say the diagnosis, medical causes or specialist terms.
7. Your role is TO BE PATIENT. Do not step out of role under any circumstances.
""".strip()



def build_system_prompt_for_case(disease_key: str, mapping: Dict[str, set]) -> str:
    """
    Construiește system prompt-ul final pentru un caz, pe baza:
      - cheii bolii (ex: 'carie_simpla')
      - mapping-ului boala -> set de simptome
    """
    diagnosis = disease_key.replace("_", " ")
    symptoms = sorted(list(mapping.get(disease_key, [])))

    if symptoms:
        symptom_bullets = "\n".join(f"- {s}" for s in symptoms)
    else:
        symptom_bullets = "- (nu sunt definite simptome pentru acest caz)"

    return SYSTEM_PROMPT_TEMPLATE.format(
        diagnosis=diagnosis,
        symptom_bullets=symptom_bullets,
    )


In [6]:
from dataclasses import dataclass, field
import random

@dataclass
class Case:
    diagnosis_truth: str
    symptoms_truth: Set[str]
    revealed_symptoms: Set[str] = field(default_factory=set)


@dataclass
class State:
    history: List[Dict[str, str]] = field(default_factory=list)


def init_case(mapping: Dict[str, set]) -> Case:
    disease_key = random.choice(list(mapping.keys()))
    symptoms = set(mapping[disease_key])
    return Case(
        diagnosis_truth=disease_key,
        symptoms_truth=symptoms,
    )


In [7]:
def step(case: Case, state: State, user_msg: str) -> str:
    state.history.append({"role": "user", "content": user_msg})

    disease_key = case.diagnosis_truth
    system_prompt = build_system_prompt_for_case(disease_key, mapping)

    messages = [{"role": "system", "content": system_prompt}] + state.history

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=80,
          min_new_tokens=10,
          do_sample=True,
          top_p=0.6,
          temperature=0.15,
          repetition_penalty=1.05,
      )


    generated_tokens = outputs[0][input_len:]
    reply = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    state.history.append({"role": "assistant", "content": reply})

    for s in case.symptoms_truth:
        if s.lower() in reply.lower():
            case.revealed_symptoms.add(s)

    return reply


In [8]:
case: Case = None
state: State = None

def start_new_case():
    """Pornește un caz nou cu diagnostic aleator."""
    global case, state
    case = init_case(mapping)
    state = State()
    print("Caz nou inceput!")
    print("Scrie /help pentru lista de comenzi.")
    # print(f"(debug) Boala interna aleasa: {case.diagnosis_truth}")
    return case, state

HELP_TEXT = """Comenzi disponibile:
/help       - afiseaza aceasta lista
/truth      - afiseaza diagnosticul real + simptomele (debug)
/revealed   - arata simptomele reale deja dezvaluite
/new        - porneste un caz nou
/exit       - inchide sesiunea de chat
"""

def show_truth():
    print("Diagnostic ASCUNS:", case.diagnosis_truth)
    print("Simptome reale:", ", ".join(sorted(case.symptoms_truth)))

def show_revealed():
    print("REVEALED:", sorted(case.revealed_symptoms) or "(niciunul)")


def chat_loop():
    print("CONVERSATIE LIVE CU PACIENTUL")
    print("Scrie intrebarile tale (sau /help)")

    while True:
        try:
            user_msg = input("\nTu: ").strip()
        except EOFError:
            break
        if not user_msg:
            continue

        cmd = user_msg.lower()
        if cmd == "/help":
            print(HELP_TEXT); continue
        if cmd == "/exit":
            print("Inchis"); break
        if cmd == "/truth":
            show_truth(); continue
        if cmd == "/revealed":
            show_revealed(); continue
        if cmd == "/new":
            start_new_case(); continue

        try:
            raspuns = step(case, state, user_msg)
            print("Pacient:", raspuns)
        except Exception as e:
            print("Eroare in step():", e)


start_new_case()
chat_loop()

Caz nou inceput!
Scrie /help pentru lista de comenzi.
CONVERSATIE LIVE CU PACIENTUL
Scrie intrebarile tale (sau /help)

Tu: hello


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Pacient: Hi. I'm here for a check-up.

Tu: /help
Comenzi disponibile:
/help       - afiseaza aceasta lista
/truth      - afiseaza diagnosticul real + simptomele (debug)
/revealed   - arata simptomele reale deja dezvaluite
/new        - porneste un caz nou
/exit       - inchide sesiunea de chat


Tu: /truth
Diagnostic ASCUNS: simple_cavities
Simptome reale: discomfort when brushing, pain to cold, pain to sour, pain to stimuli, pain to sweet, short sharp pain

Tu: do you have any discomfort?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Pacient: When I brush my teeth, I feel a bit uncomfortable.

Tu: do you have any pain to warm water?


...

Pacient: Yes, cold water makes me feel some pain.

Tu: /end


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Pacient: What's going on? Did we finish already?

Tu: /ext


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Pacient: I think there was a mistake. Let's continue.

Tu: /exit 
Inchis


Metrics

In [14]:
"""
Evaluation Metrics for Dental Patient Simulation LLM Agent
Implements SFS, RPCS, and CRS metrics
"""

from dataclasses import dataclass
from typing import List, Set, Dict, Tuple
import re
from collections import defaultdict


@dataclass
class EvaluationResult:
    """Container for evaluation results"""
    sfs: float  # Symptom Fidelity Score
    rpcs: float  # Role-Playing Consistency Score
    crs: float  # Clinical Realism Score
    details: Dict[str, any]


class SymptomFidelityEvaluator:
    """
    Evaluates how consistently the LLM uses symptoms from knowledge base
    SFS = 1 - (incorrect or missing symptoms / total expected symptoms)
    """

    def __init__(self, mapping: Dict[str, Set[str]]):
        self.mapping = mapping

    def extract_symptoms_from_text(self, text: str, symptom_list: Set[str]) -> Set[str]:
        """Extract mentioned symptoms from LLM output"""
        text_lower = text.lower()
        mentioned = set()

        for symptom in symptom_list:
            # Check for symptom or close variations
            symptom_lower = symptom.lower()
            if symptom_lower in text_lower:
                mentioned.add(symptom)
            # Check for partial matches (e.g., "pain to cold" matches "cold")
            elif any(word in text_lower for word in symptom_lower.split()):
                mentioned.add(symptom)

        return mentioned

    def evaluate(self,
                 diagnosis_key: str,
                 conversation_history: List[Dict[str, str]],
                 revealed_symptoms: Set[str]) -> Tuple[float, Dict]:
        """
        Calculate Symptom Fidelity Score

        Returns:
            (score, details_dict)
        """
        expected_symptoms = self.mapping.get(diagnosis_key, set())

        if not expected_symptoms:
            return 1.0, {"error": "No symptoms defined for diagnosis"}

        # Extract all patient responses
        patient_responses = [
            turn['content'] for turn in conversation_history
            if turn['role'] == 'assistant'
        ]
        full_text = " ".join(patient_responses)

        # Find mentioned symptoms
        mentioned = self.extract_symptoms_from_text(full_text, expected_symptoms)

        # Calculate hallucinations (symptoms mentioned but not in expected)
        # Extract potential symptoms using heuristics
        hallucinated = self._detect_hallucinations(full_text, expected_symptoms)

        # Calculate metrics
        correct = len(mentioned & expected_symptoms)
        missing = len(expected_symptoms - mentioned)
        incorrect = len(hallucinated)

        total_errors = missing + incorrect
        sfs = max(0.0, 1.0 - (total_errors / len(expected_symptoms)))

        details = {
            "expected_count": len(expected_symptoms),
            "correctly_mentioned": correct,
            "missing_symptoms": list(expected_symptoms - mentioned),
            "hallucinated_symptoms": list(hallucinated),
            "revealed_symptoms": list(revealed_symptoms)
        }

        return sfs, details

    def _detect_hallucinations(self, text: str, expected: Set[str]) -> Set[str]:
        """Detect symptoms mentioned that aren't in expected set"""
        # Common dental symptom keywords that might indicate hallucination
        symptom_patterns = [
            r"bleeding", r"fever", r"headache", r"nausea",
            r"discharge", r"numbness", r"tingling"
        ]

        hallucinated = set()
        text_lower = text.lower()

        for pattern in symptom_patterns:
            if re.search(pattern, text_lower):
                # Check if this symptom is in expected
                if not any(pattern in str(s).lower() for s in expected):
                    hallucinated.add(pattern)

        return hallucinated


class RolePlayingConsistencyEvaluator:
    """
    Evaluates whether LLM maintains patient persona
    1.0 = Always patient, 0.5 = Occasional breaks, 0.0 = Frequent breaks
    """

    # Patterns indicating persona breaking
    DOCTOR_PATTERNS = [
        r"diagnosis", r"diagnose", r"medical condition",
        r"treatment", r"you should see", r"i recommend",
        r"this could be", r"it might be", r"probably",
        r"consult", r"examination needed"
    ]

    META_PATTERNS = [
        r"as an ai", r"i am a language model", r"i don't have",
        r"i cannot", r"system prompt", r"my role"
    ]

    MEDICAL_ADVICE = [
        r"take.*medication", r"antibiotic", r"prescription",
        r"dental procedure", r"root canal", r"extraction"
    ]

    def evaluate(self, conversation_history: List[Dict[str, str]]) -> Tuple[float, Dict]:
        """
        Calculate Role-Playing Consistency Score

        Returns:
            (score, details_dict)
        """
        patient_responses = [
            turn['content'] for turn in conversation_history
            if turn['role'] == 'assistant'
        ]

        if not patient_responses:
            return 1.0, {"error": "No patient responses found"}

        violations = {
            "doctor_language": 0,
            "meta_awareness": 0,
            "medical_advice": 0
        }

        violation_examples = defaultdict(list)

        for response in patient_responses:
            response_lower = response.lower()

            # Check for doctor language
            for pattern in self.DOCTOR_PATTERNS:
                matches = re.findall(pattern, response_lower)
                if matches:
                    violations["doctor_language"] += len(matches)
                    violation_examples["doctor_language"].append(
                        f"'{response[:50]}...' contains '{matches[0]}'"
                    )

            # Check for meta-awareness
            for pattern in self.META_PATTERNS:
                matches = re.findall(pattern, response_lower)
                if matches:
                    violations["meta_awareness"] += len(matches)
                    violation_examples["meta_awareness"].append(
                        f"'{response[:50]}...' contains '{matches[0]}'"
                    )

            # Check for medical advice
            for pattern in self.MEDICAL_ADVICE:
                matches = re.findall(pattern, response_lower)
                if matches:
                    violations["medical_advice"] += len(matches)
                    violation_examples["medical_advice"].append(
                        f"'{response[:50]}...' contains '{matches[0]}'"
                    )

        total_violations = sum(violations.values())
        total_responses = len(patient_responses)

        # Calculate score
        if total_violations == 0:
            rpcs = 1.0
        elif total_violations <= total_responses * 0.1:  # Less than 10% violation rate
            rpcs = 0.8
        elif total_violations <= total_responses * 0.3:  # 10-30% violation rate
            rpcs = 0.5
        else:
            rpcs = 0.2

        details = {
            "total_responses": total_responses,
            "violations": violations,
            "examples": dict(violation_examples),
            "violation_rate": total_violations / total_responses
        }

        return rpcs, details


class ClinicalRealismEvaluator:
    """
    Evaluates realism of patient responses
    Scores 0-4 on multiple dimensions, normalized to 0-1
    """

    # Positive indicators (realistic patient language)
    PATIENT_LANGUAGE = [
        r"hurts?", r"pain", r"ache", r"sore", r"uncomfortable",
        r"feels?", r"it's", r"when i", r"sometimes", r"usually"
    ]

    # Negative indicators (overly medical)
    MEDICAL_JARGON = [
        r"periodontitis", r"pulpitis", r"necrosis", r"hyperplasia",
        r"pathology", r"etiology", r"prognosis"
    ]

    def evaluate(self, conversation_history: List[Dict[str, str]]) -> Tuple[float, Dict]:
        """
        Calculate Clinical Realism Score

        Dimensions:
        1. Symptom description realism
        2. Appropriate vocabulary
        3. Pain descriptions
        4. Consistency with disease

        Returns:
            (score, details_dict)
        """
        patient_responses = [
            turn['content'] for turn in conversation_history
            if turn['role'] == 'assistant'
        ]

        if not patient_responses:
            return 1.0, {"error": "No patient responses found"}

        full_text = " ".join(patient_responses)
        text_lower = full_text.lower()

        # Dimension 1: Symptom description realism (0-4)
        symptom_score = self._score_symptom_descriptions(patient_responses)

        # Dimension 2: Vocabulary appropriateness (0-4)
        vocab_score = self._score_vocabulary(text_lower)

        # Dimension 3: Pain descriptions (0-4)
        pain_score = self._score_pain_descriptions(text_lower)

        # Dimension 4: Consistency (0-4)
        consistency_score = self._score_consistency(patient_responses)

        # Normalize to 0-1
        total_score = (symptom_score + vocab_score + pain_score + consistency_score) / 16.0

        details = {
            "symptom_description": symptom_score / 4.0,
            "vocabulary": vocab_score / 4.0,
            "pain_description": pain_score / 4.0,
            "consistency": consistency_score / 4.0,
            "dimensions_raw": {
                "symptom": symptom_score,
                "vocab": vocab_score,
                "pain": pain_score,
                "consistency": consistency_score
            }
        }

        return total_score, details

    def _score_symptom_descriptions(self, responses: List[str]) -> int:
        """Score how realistically symptoms are described (0-4)"""
        score = 0

        # Check for subjective descriptions
        subjective_count = sum(
            1 for r in responses
            if any(p in r.lower() for p in ["i feel", "it feels", "feels like"])
        )
        if subjective_count > 0:
            score += 2

        # Check for concrete, relatable descriptions
        concrete_count = sum(
            1 for r in responses
            if any(p in r.lower() for p in ["when i", "after i", "if i"])
        )
        if concrete_count > 0:
            score += 2

        return min(4, score)

    def _score_vocabulary(self, text: str) -> int:
        """Score appropriateness of vocabulary (0-4)"""
        # Count patient-appropriate language
        patient_lang = sum(
            len(re.findall(pattern, text))
            for pattern in self.PATIENT_LANGUAGE
        )

        # Count medical jargon (negative)
        jargon = sum(
            len(re.findall(pattern, text))
            for pattern in self.MEDICAL_JARGON
        )

        if jargon > 2:
            return 0  # Too much jargon
        elif jargon > 0:
            return 2  # Some jargon
        elif patient_lang >= 5:
            return 4  # Excellent patient language
        elif patient_lang >= 2:
            return 3  # Good patient language
        else:
            return 1

    def _score_pain_descriptions(self, text: str) -> int:
        """Score quality of pain descriptions (0-4)"""
        score = 0

        # Location specificity
        if re.search(r"(back|front|side|top|bottom|left|right)", text):
            score += 1

        # Intensity descriptors
        if re.search(r"(sharp|dull|throbbing|constant|mild|severe)", text):
            score += 1

        # Trigger descriptions
        if re.search(r"(when|after|during|while)", text):
            score += 1

        # Duration mentions
        if re.search(r"(minute|hour|day|week|always|sometimes)", text):
            score += 1

        return score

    def _score_consistency(self, responses: List[str]) -> int:
        """Score consistency across responses (0-4)"""
        if len(responses) < 2:
            return 4  # Can't check consistency with one response

        # Check for contradictions (simple heuristic)
        # This is a placeholder - could be made more sophisticated
        score = 4

        # Check if pain descriptions are consistent
        pain_keywords = set()
        for response in responses:
            response_lower = response.lower()
            if "pain" in response_lower:
                # Extract pain descriptors
                for word in ["sharp", "dull", "throbbing", "aching"]:
                    if word in response_lower:
                        pain_keywords.add(word)

        # If more than 2 different pain types mentioned, might be inconsistent
        if len(pain_keywords) > 2:
            score -= 1

        return max(0, score)


class PatientSimulationEvaluator:
    """Main evaluator combining all metrics"""

    def __init__(self, mapping: Dict[str, Set[str]]):
        self.sfs_evaluator = SymptomFidelityEvaluator(mapping)
        self.rpcs_evaluator = RolePlayingConsistencyEvaluator()
        self.crs_evaluator = ClinicalRealismEvaluator()

    def evaluate(self,
                 case,
                 state) -> EvaluationResult:
        """
        Evaluate a complete conversation

        Args:
            case: Case object with diagnosis_truth and symptoms_truth
            state: State object with conversation history

        Returns:
            EvaluationResult with all metrics
        """
        # Calculate SFS
        sfs, sfs_details = self.sfs_evaluator.evaluate(
            case.diagnosis_truth,
            state.history,
            case.revealed_symptoms
        )

        # Calculate RPCS
        rpcs, rpcs_details = self.rpcs_evaluator.evaluate(state.history)

        # Calculate CRS
        crs, crs_details = self.crs_evaluator.evaluate(state.history)

        # Combine details
        all_details = {
            "sfs": sfs_details,
            "rpcs": rpcs_details,
            "crs": crs_details,
            "conversation_length": len(state.history),
            "diagnosis": case.diagnosis_truth
        }

        return EvaluationResult(
            sfs=sfs,
            rpcs=rpcs,
            crs=crs,
            details=all_details
        )

    def print_report(self, result: EvaluationResult):
        """Print a formatted evaluation report"""
        print("\n" + "="*60)
        print("PATIENT SIMULATION EVALUATION REPORT")
        print("="*60)

        print(f"\n📊 OVERALL SCORES:")
        print(f"   Symptom Fidelity Score (SFS):      {result.sfs:.3f}")
        print(f"   Role-Playing Consistency (RPCS):   {result.rpcs:.3f}")
        print(f"   Clinical Realism Score (CRS):      {result.crs:.3f}")
        print(f"   Average Score:                      {(result.sfs + result.rpcs + result.crs) / 3:.3f}")

        print(f"\n🔍 DETAILED BREAKDOWN:")

        # SFS details
        print(f"\n   Symptom Fidelity:")
        sfs_d = result.details['sfs']
        print(f"   - Expected symptoms: {sfs_d['expected_count']}")
        print(f"   - Correctly mentioned: {sfs_d['correctly_mentioned']}")
        if sfs_d['missing_symptoms']:
            print(f"   - Missing: {', '.join(sfs_d['missing_symptoms'][:3])}")
        if sfs_d['hallucinated_symptoms']:
            print(f"   - Hallucinated: {', '.join(sfs_d['hallucinated_symptoms'])}")

        # RPCS details
        print(f"\n   Role-Playing Consistency:")
        rpcs_d = result.details['rpcs']
        print(f"   - Total responses: {rpcs_d['total_responses']}")
        print(f"   - Violation rate: {rpcs_d['violation_rate']:.1%}")
        for vtype, count in rpcs_d['violations'].items():
            if count > 0:
                print(f"   - {vtype}: {count} violations")

        # CRS details
        print(f"\n   Clinical Realism:")
        crs_d = result.details['crs']
        print(f"   - Symptom description: {crs_d['symptom_description']:.2f}")
        print(f"   - Vocabulary: {crs_d['vocabulary']:.2f}")
        print(f"   - Pain description: {crs_d['pain_description']:.2f}")
        print(f"   - Consistency: {crs_d['consistency']:.2f}")

        print("\n" + "="*60 + "\n")


# Example usage function
def evaluate_conversation(case, state, mapping):
    """
    Convenience function to evaluate a conversation

    Args:
        case: Case object from your simulation
        state: State object with conversation history
        mapping: The disease->symptoms mapping dictionary

    Returns:
        EvaluationResult
    """
    evaluator = PatientSimulationEvaluator(mapping)
    result = evaluator.evaluate(case, state)
    evaluator.print_report(result)
    return result

Tests

In [15]:
"""
Comprehensive Test Suite for Dental Patient Simulation LLM
Run this in your Colab notebook after loading the model
"""

import random
from typing import List, Dict, Tuple
import json
from datetime import datetime

# ============================================================================
# REALISTIC DENTAL CONSULTATION QUESTIONS
# ============================================================================
CONSULTATION_QUESTIONS = {
    "opening": [
        "Good morning! What brings you to the dental clinic today?",
        "Hello, I'm Dr. Smith. Can you tell me what's been bothering you?",
        "Hi there, what seems to be the problem with your teeth?",
        "Welcome to the clinic. What can I help you with today?",
    ],

    "chief_complaint": [
        "Can you describe the pain you're experiencing in more detail?",
        "Where exactly in your mouth is the pain located? Can you point to the specific tooth or area?",
        "How long have you been experiencing these symptoms?",
        "When did you first notice this problem?",
        "Is this the first time you've had this issue, or has it happened before?",
    ],

    "pain_characteristics": [
        "On a scale of 1 to 10, with 10 being the worst pain imaginable, how would you rate your pain right now?",
        "How would you describe the pain? Is it sharp, dull, throbbing, constant, or does it come and go?",
        "Does the pain spread to other areas, like your jaw, ear, or head?",
        "Is the pain worse at any particular time of day - morning, afternoon, or night?",
        "Does the pain wake you up at night, or does it affect your sleep?",
    ],

    "triggers_and_relievers": [
        "What makes the pain worse? For example, eating, drinking, or touching the area?",
        "Does hot food or drinks cause any problems? What about cold things like ice cream or cold water?",
        "Do sweet foods or drinks trigger the pain or make it worse?",
        "What about chewing - does biting down on food cause pain?",
        "Have you found anything that helps relieve the pain or make it more bearable?",
        "Does the pain get better when you take painkillers like ibuprofen or paracetamol?",
    ],

    "associated_symptoms": [
        "Have you noticed any swelling in your gums or face?",
        "Is there any bleeding when you brush your teeth or eat?",
        "Do you have any unusual taste in your mouth, perhaps a bad or salty taste?",
        "Have you noticed any bad breath or odor coming from your mouth?",
        "Is there any discharge or pus coming from your gums?",
        "Have you had any fever or felt generally unwell?",
        "Do you have any difficulty opening your mouth fully?",
        "Does it hurt when you swallow or move your jaw?",
    ],

    "specific_observations": [
        "Have you noticed if the tooth feels loose or moves when you touch it?",
        "Does the tooth feel higher than your other teeth when you bite down?",
        "Can you see any visible holes, dark spots, or broken pieces on your teeth?",
        "Have you noticed any changes in the color of your teeth or gums?",
        "Is there a particular tooth that hurts when you tap on it with your finger?",
    ],

    "impact_on_daily_life": [
        "Is the pain affecting your ability to eat or drink normally?",
        "Are you avoiding chewing on one side of your mouth because of the pain?",
        "Has this problem affected your work or daily activities?",
        "Are you able to brush your teeth normally, or does that area hurt too much?",
    ],

    "follow_up": [
        "Is there anything else that you've noticed or that's concerning you?",
        "Are there any other symptoms you'd like to tell me about?",
        "Have you tried anything at home to help with this problem?",
        "Is there anything you'd like to ask me about your symptoms?",
    ]
}

# ============================================================================
# DISEASE-SPECIFIC QUESTION SETS
# ============================================================================
DISEASE_SPECIFIC_QUESTIONS = {
    "periodontal_abscess": [
        "Is there a specific area on your gum that looks swollen or feels like a bump?",
        "When you press on the swollen area, does it hurt more?",
        "Have you noticed any liquid or drainage coming from your gums?",
        "Does it feel like the pain is coming from the gum rather than the tooth itself?",
    ],

    "simple_cavities": [
        "Does the pain only last for a few seconds when you eat or drink something cold or sweet?",
        "After the painful sensation, does it go away quickly or linger?",
        "Can you see any dark spots or holes in your teeth when you look in the mirror?",
    ],

    "acute_apical_periodontitis": [
        "Does it hurt specifically when you bite down or chew food?",
        "When you tap the tooth with your finger, is it very sensitive?",
        "Does the tooth feel like it's sitting higher or sticking out more than your other teeth?",
        "Is the tooth loose or does it move slightly when you touch it?",
    ],

    "pericoronitis": [
        "Is the pain near your wisdom tooth at the back of your mouth?",
        "Can you open your mouth all the way, or is it difficult?",
        "Is it painful when you swallow food or saliva?",
        "Can you see the gum covering part of your wisdom tooth, and does it look red or swollen?",
    ],

    "reversible_pulpitis": [
        "Does the pain only happen when something cold touches the tooth?",
        "Once you remove the cold drink or food, does the pain stop immediately?",
        "Do you ever have pain when you're not eating or drinking - like spontaneous pain?",
    ],

    "irreversible_pulpitis": [
        "Do you get sudden, severe pain that happens even when you're not eating or drinking?",
        "Does the pain last for several minutes or longer after it starts?",
        "Does heat make it worse - like drinking hot coffee or tea?",
        "Does the pain sometimes shoot up to your ear or spread to your head?",
        "Does cold actually help relieve the pain, or does it make it worse?",
    ],

    "pulp_necrosis": [
        "Did you have severe pain before, but now the pain has changed or become different?",
        "Does heat cause severe, immediate pain that lasts a while?",
        "When you bite down, does it feel like there's pressure building up in the tooth?",
        "If I were to tap on the tooth, would that be very painful?",
    ]
}

# ============================================================================
# TEST CONVERSATION BUILDER
# ============================================================================
class TestConversationBuilder:
    """Builds realistic test conversations"""

    def __init__(self):
        self.question_pool = CONSULTATION_QUESTIONS
        self.disease_questions = DISEASE_SPECIFIC_QUESTIONS

    def build_short_conversation(self, disease_key: str = None) -> List[str]:
        """Build a short 5-7 question conversation"""
        questions = []

        # Opening
        questions.append(random.choice(self.question_pool["opening"]))

        # Chief complaint
        questions.append(random.choice(self.question_pool["chief_complaint"]))

        # Pain characteristics
        questions.append(random.choice(self.question_pool["pain_characteristics"]))

        # Triggers
        questions.append(random.choice(self.question_pool["triggers_and_relievers"]))

        # Associated symptoms
        questions.append(random.choice(self.question_pool["associated_symptoms"]))

        # Disease-specific if provided
        if disease_key and disease_key in self.disease_questions:
            questions.append(random.choice(self.disease_questions[disease_key]))

        # Follow-up
        questions.append(random.choice(self.question_pool["follow_up"]))

        return questions

    def build_medium_conversation(self, disease_key: str = None) -> List[str]:
        """Build a medium 10-12 question conversation"""
        questions = []

        # Opening
        questions.append(random.choice(self.question_pool["opening"]))

        # Chief complaint (2 questions)
        questions.extend(random.sample(self.question_pool["chief_complaint"], 2))

        # Pain characteristics (2 questions)
        questions.extend(random.sample(self.question_pool["pain_characteristics"], 2))

        # Triggers (2 questions)
        questions.extend(random.sample(self.question_pool["triggers_and_relievers"], 2))

        # Associated symptoms (2 questions)
        questions.extend(random.sample(self.question_pool["associated_symptoms"], 2))

        # Specific observations
        questions.append(random.choice(self.question_pool["specific_observations"]))

        # Disease-specific if provided
        if disease_key and disease_key in self.disease_questions:
            questions.extend(random.sample(self.disease_questions[disease_key], min(2, len(self.disease_questions[disease_key]))))

        # Follow-up
        questions.append(random.choice(self.question_pool["follow_up"]))

        return questions

    def build_long_conversation(self, disease_key: str = None) -> List[str]:
        """Build a long 15-20 question conversation"""
        questions = []

        # Opening
        questions.append(random.choice(self.question_pool["opening"]))

        # Comprehensive coverage
        for phase in ["chief_complaint", "pain_characteristics", "triggers_and_relievers",
                      "associated_symptoms", "specific_observations", "impact_on_daily_life"]:
            num_questions = 3 if phase in ["pain_characteristics", "triggers_and_relievers", "associated_symptoms"] else 2
            available = self.question_pool[phase]
            questions.extend(random.sample(available, min(num_questions, len(available))))

        # Disease-specific if provided
        if disease_key and disease_key in self.disease_questions:
            questions.extend(self.disease_questions[disease_key])

        # Follow-up (2 questions)
        questions.extend(random.sample(self.question_pool["follow_up"], 2))

        return questions

# ============================================================================
# AUTOMATED TEST RUNNER
# ============================================================================

class PatientSimulationTester:
    """Automated tester for patient simulation LLM"""

    def __init__(self, model, tokenizer, mapping, step_function):
        """
        Args:
            model: Your loaded LLM model
            tokenizer: Tokenizer for the model
            mapping: Disease -> symptoms mapping
            step_function: Your step() function from notebook
        """
        self.model = model
        self.tokenizer = tokenizer
        self.mapping = mapping
        self.step = step_function
        self.conversation_builder = TestConversationBuilder()

    def run_single_test(self,
                       disease_key: str = None,
                       conversation_length: str = "medium",
                       verbose: bool = True) -> Tuple[any, any]:
        """
        Run a single test conversation

        Args:
            disease_key: Specific disease to test (None for random)
            conversation_length: "short", "medium", or "long"
            verbose: Print conversation as it happens

        Returns:
            (case, state) tuple
        """
        from dataclasses import dataclass, field
        from typing import Set

        # Import Case and State from your notebook
        @dataclass
        class Case:
            diagnosis_truth: str
            symptoms_truth: Set[str]
            revealed_symptoms: Set[str] = field(default_factory=set)

        @dataclass
        class State:
            history: List[Dict[str, str]] = field(default_factory=list)
            memory_summary: str = ""
            turns_since_summary: int = 0

        # Initialize case
        if disease_key is None:
            disease_key = random.choice(list(self.mapping.keys()))

        symptoms = set(self.mapping[disease_key])
        case = Case(
            diagnosis_truth=disease_key,
            symptoms_truth=symptoms,
        )
        state = State()

        # Build question set
        if conversation_length == "short":
            questions = self.conversation_builder.build_short_conversation(disease_key)
        elif conversation_length == "long":
            questions = self.conversation_builder.build_long_conversation(disease_key)
        else:  # medium
            questions = self.conversation_builder.build_medium_conversation(disease_key)

        if verbose:
            print(f"\n{'='*70}")
            print(f"Testing: {disease_key.replace('_', ' ').title()}")
            print(f"Expected symptoms: {len(symptoms)}")
            print(f"Conversation length: {conversation_length} ({len(questions)} questions)")
            print(f"{'='*70}\n")

        # Run conversation
        for i, question in enumerate(questions, 1):
            if verbose:
                print(f"[Q{i}] Doctor: {question}")

            try:
                response = self.step(case, state, question)

                if verbose:
                    print(f"[A{i}] Patient: {response}\n")

            except Exception as e:
                print(f"❌ Error at question {i}: {e}")
                break

        return case, state

    def run_test_suite(self,
                      num_tests_per_disease: int = 2,
                      conversation_length: str = "medium",
                      save_results: bool = True) -> Dict:
        """
        Run comprehensive test suite across all diseases

        Args:
            num_tests_per_disease: Number of conversations per disease
            conversation_length: "short", "medium", or "long"
            save_results: Save results to JSON file

        Returns:
            Dictionary with all results
        """
        from metrics import PatientSimulationEvaluator

        evaluator = PatientSimulationEvaluator(self.mapping)

        all_results = {
            "test_info": {
                "timestamp": datetime.now().isoformat(),
                "num_tests_per_disease": num_tests_per_disease,
                "conversation_length": conversation_length,
                "total_diseases": len(self.mapping)
            },
            "disease_results": {},
            "aggregate_metrics": {}
        }

        print(f"\n{'='*70}")
        print(f"STARTING TEST SUITE")
        print(f"{'='*70}")
        print(f"Testing {len(self.mapping)} diseases")
        print(f"{num_tests_per_disease} conversations per disease")
        print(f"Conversation length: {conversation_length}")
        print(f"Total tests: {len(self.mapping) * num_tests_per_disease}\n")

        all_scores = {"sfs": [], "rpcs": [], "crs": []}

        for disease_idx, disease_key in enumerate(self.mapping.keys(), 1):
            print(f"\n[{disease_idx}/{len(self.mapping)}] Testing: {disease_key.replace('_', ' ').title()}")
            print("-" * 70)

            disease_results = []

            for test_num in range(num_tests_per_disease):
                print(f"\n  Test {test_num + 1}/{num_tests_per_disease}...")

                # Run test
                case, state = self.run_single_test(
                    disease_key=disease_key,
                    conversation_length=conversation_length,
                    verbose=False
                )

                # Evaluate
                result = evaluator.evaluate(case, state)

                # Store
                disease_results.append({
                    "sfs": result.sfs,
                    "rpcs": result.rpcs,
                    "crs": result.crs,
                    "details": result.details
                })

                # Aggregate
                all_scores["sfs"].append(result.sfs)
                all_scores["rpcs"].append(result.rpcs)
                all_scores["crs"].append(result.crs)

                print(f"    SFS: {result.sfs:.3f} | RPCS: {result.rpcs:.3f} | CRS: {result.crs:.3f}")

            # Calculate disease averages
            avg_sfs = sum(r["sfs"] for r in disease_results) / len(disease_results)
            avg_rpcs = sum(r["rpcs"] for r in disease_results) / len(disease_results)
            avg_crs = sum(r["crs"] for r in disease_results) / len(disease_results)

            all_results["disease_results"][disease_key] = {
                "average_scores": {
                    "sfs": avg_sfs,
                    "rpcs": avg_rpcs,
                    "crs": avg_crs,
                    "overall": (avg_sfs + avg_rpcs + avg_crs) / 3
                },
                "individual_tests": disease_results
            }

            print(f"\n  Disease Average: SFS={avg_sfs:.3f}, RPCS={avg_rpcs:.3f}, CRS={avg_crs:.3f}")

        # Calculate overall aggregate metrics
        all_results["aggregate_metrics"] = {
            "sfs": {
                "mean": sum(all_scores["sfs"]) / len(all_scores["sfs"]),
                "min": min(all_scores["sfs"]),
                "max": max(all_scores["sfs"]),
                "std": self._std(all_scores["sfs"])
            },
            "rpcs": {
                "mean": sum(all_scores["rpcs"]) / len(all_scores["rpcs"]),
                "min": min(all_scores["rpcs"]),
                "max": max(all_scores["rpcs"]),
                "std": self._std(all_scores["rpcs"])
            },
            "crs": {
                "mean": sum(all_scores["crs"]) / len(all_scores["crs"]),
                "min": min(all_scores["crs"]),
                "max": max(all_scores["crs"]),
                "std": self._std(all_scores["crs"])
            }
        }

        # Print final report
        self._print_final_report(all_results)

        # Save results
        if save_results:
            filename = f"test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(filename, 'w') as f:
                json.dump(all_results, f, indent=2, default=str)
            print(f"\n✅ Results saved to: {filename}")

        return all_results

    def _std(self, values):
        """Calculate standard deviation"""
        import math
        if len(values) == 0:
            return 0
        mean = sum(values) / len(values)
        variance = sum((x - mean) ** 2 for x in values) / len(values)
        return math.sqrt(variance)

    def _print_final_report(self, results):
        """Print comprehensive final report"""
        print(f"\n\n{'='*70}")
        print("FINAL TEST SUITE REPORT")
        print(f"{'='*70}\n")

        agg = results["aggregate_metrics"]

        print("📊 OVERALL PERFORMANCE:")
        print(f"   Symptom Fidelity Score (SFS)")
        print(f"      Mean: {agg['sfs']['mean']:.3f} ± {agg['sfs']['std']:.3f}")
        print(f"      Range: [{agg['sfs']['min']:.3f}, {agg['sfs']['max']:.3f}]")
        print()
        print(f"   Role-Playing Consistency (RPCS)")
        print(f"      Mean: {agg['rpcs']['mean']:.3f} ± {agg['rpcs']['std']:.3f}")
        print(f"      Range: [{agg['rpcs']['min']:.3f}, {agg['rpcs']['max']:.3f}]")
        print()
        print(f"   Clinical Realism Score (CRS)")
        print(f"      Mean: {agg['crs']['mean']:.3f} ± {agg['crs']['std']:.3f}")
        print(f"      Range: [{agg['crs']['min']:.3f}, {agg['crs']['max']:.3f}]")

        overall_avg = (agg['sfs']['mean'] + agg['rpcs']['mean'] + agg['crs']['mean']) / 3
        print(f"\n   🎯 OVERALL AVERAGE: {overall_avg:.3f}")

        # Best and worst diseases
        print(f"\n\n📈 PERFORMANCE BY DISEASE:\n")

        disease_scores = []
        for disease, data in results["disease_results"].items():
            score = data["average_scores"]["overall"]
            disease_scores.append((disease, score, data["average_scores"]))

        disease_scores.sort(key=lambda x: x[1], reverse=True)

        print("   Top 3 Best Performing:")
        for i, (disease, score, scores) in enumerate(disease_scores[:3], 1):
            print(f"      {i}. {disease.replace('_', ' ').title()}: {score:.3f}")
            print(f"         (SFS: {scores['sfs']:.3f}, RPCS: {scores['rpcs']:.3f}, CRS: {scores['crs']:.3f})")

        print("\n   Bottom 3 Need Improvement:")
        for i, (disease, score, scores) in enumerate(disease_scores[-3:], 1):
            print(f"      {i}. {disease.replace('_', ' ').title()}: {score:.3f}")
            print(f"         (SFS: {scores['sfs']:.3f}, RPCS: {scores['rpcs']:.3f}, CRS: {scores['crs']:.3f})")

        print(f"\n{'='*70}\n")


# ============================================================================
# QUICK START FUNCTIONS
# ============================================================================

def quick_test(model, tokenizer, mapping, step_function,
               num_conversations: int = 5):
    """
    Quick test - runs a few conversations and shows results

    Usage:
        quick_test(model, tokenizer, mapping, step, num_conversations=5)
    """
    tester = PatientSimulationTester(model, tokenizer, mapping, step_function)

    print("🚀 Running Quick Test...")
    print(f"   {num_conversations} random conversations\n")

    from metrics import PatientSimulationEvaluator
    evaluator = PatientSimulationEvaluator(mapping)

    results = []
    for i in range(num_conversations):
        case, state = tester.run_single_test(verbose=True)
        result = evaluator.evaluate(case, state)
        evaluator.print_report(result)
        results.append(result)

    # Summary
    avg_sfs = sum(r.sfs for r in results) / len(results)
    avg_rpcs = sum(r.rpcs for r in results) / len(results)
    avg_crs = sum(r.crs for r in results) / len(results)

    print(f"\n{'='*70}")
    print(f"QUICK TEST SUMMARY")
    print(f"{'='*70}")
    print(f"Average SFS:  {avg_sfs:.3f}")
    print(f"Average RPCS: {avg_rpcs:.3f}")
    print(f"Average CRS:  {avg_crs:.3f}")
    print(f"Overall:      {(avg_sfs + avg_rpcs + avg_crs) / 3:.3f}")
    print(f"{'='*70}\n")


def full_test_suite(model, tokenizer, mapping, step_function,
                   tests_per_disease: int = 3,
                   length: str = "medium"):
    """
    Full test suite - comprehensive evaluation of all diseases

    Usage:
        results = full_test_suite(model, tokenizer, mapping, step,
                                 tests_per_disease=3, length="medium")
    """
    tester = PatientSimulationTester(model, tokenizer, mapping, step_function)
    return tester.run_test_suite(
        num_tests_per_disease=tests_per_disease,
        conversation_length=length,
        save_results=True
    )

In [16]:
"""
ADD THIS CELL TO YOUR NOTEBOOK TO RUN TESTS
Copy the evaluation metrics files first, then run this
"""
# ============================================================================
# Run a Quick Test (5 conversations)
# ============================================================================

print("="*70)
print("OPTION 1: QUICK TEST")
print("="*70)
print("Runs 5 random conversations with your LLM and evaluates them")
print()

# Uncomment to run:
quick_test(model, tokenizer, mapping, step, num_conversations=5)


# ============================================================================
# Run Full Test Suite (all diseases, multiple tests each)
# ============================================================================

print("\n" + "="*70)
print("OPTION 2: FULL TEST SUITE")
print("="*70)
print("Tests all 8 diseases with 2-3 conversations each")
print("Generates comprehensive report + saves to JSON")
print()

# Uncomment to run:
# from tests import full_test_suite
# results = full_test_suite(
#     model,
#     tokenizer,
#     mapping,
#     step,
#     tests_per_disease=2,    # 2 conversations per disease = 16 total
#     length="medium"          # "short" (5-7 Q), "medium" (10-12 Q), "long" (15-20 Q)
# )


# ============================================================================
# Test a Specific Disease
# ============================================================================

print("\n" + "="*70)
print("OPTION 3: TEST SPECIFIC DISEASE")
print("="*70)
print("Test one specific disease in detail")
print()

# Example: Test periodontal abscess
# from tests import PatientSimulationTester
# from metrics import PatientSimulationEvaluator

# tester = PatientSimulationTester(model, tokenizer, mapping, step)
# evaluator = PatientSimulationEvaluator(mapping)

# # Run test
# case, state = tester.run_single_test(
#     disease_key="periodontal_abscess",  # or any disease from mapping
#     conversation_length="long",          # "short", "medium", or "long"
#     verbose=True
# )

# # Evaluate
# result = evaluator.evaluate(case, state)
# evaluator.print_report(result)


# ============================================================================
# Custom Test with Your Own Questions
# ============================================================================

print("\n" + "="*70)
print("OPTION 4: CUSTOM QUESTIONS")
print("="*70)
print("Test with your own specific questions")
print()

# Example:
# from dataclasses import dataclass, field
# from typing import Set, List, Dict
# from metrics import PatientSimulationEvaluator

# class Case:
#     diagnosis_truth: str
#     symptoms_truth: Set[str]
#     revealed_symptoms: Set[str] = field(default_factory=set)

# class State:
#     history: List[Dict[str, str]] = field(default_factory=list)
#     memory_summary: str = ""
#     turns_since_summary: int = 0

# # Initialize
# disease_key = "irreversible_pulpitis"
# symptoms = set(mapping[disease_key])
# case = Case(diagnosis_truth=disease_key, symptoms_truth=symptoms)
# state = State()

# # Your custom questions
# my_questions = [
#     "What brings you here today?",
#     "Where exactly does it hurt?",
#     "Does cold water make it worse or better?",
#     "Do you get pain even when you're not eating?",
#     "Does the pain radiate to your ear?",
#     "How long does the pain typically last?",
#     "What have you tried to relieve the pain?"
# ]

# # Run conversation
# for question in my_questions:
#     response = step(case, state, question)
#     print(f"Q: {question}")
#     print(f"A: {response}\n")

# # Evaluate
# evaluator = PatientSimulationEvaluator(mapping)
# result = evaluator.evaluate(case, state)
# evaluator.print_report(result)


# ============================================================================
# EXAMPLE OUTPUT YOU'LL SEE
# ============================================================================

"""
When you run the tests, you'll see output like:

======================================================================
STARTING TEST SUITE
======================================================================
Testing 8 diseases
2 conversations per disease
Conversation length: medium
Total tests: 16

[1/8] Testing: Periodontal Abscess
----------------------------------------------------------------------

  Test 1/2...
    SFS: 0.875 | RPCS: 1.000 | CRS: 0.812

  Test 2/2...
    SFS: 0.750 | RPCS: 0.950 | CRS: 0.788

  Disease Average: SFS=0.812, RPCS=0.975, CRS=0.800

[2/8] Testing: Simple Cavities
----------------------------------------------------------------------
...

======================================================================
FINAL TEST SUITE REPORT
======================================================================

📊 OVERALL PERFORMANCE:
   Symptom Fidelity Score (SFS)
      Mean: 0.856 ± 0.082
      Range: [0.714, 0.950]

   Role-Playing Consistency (RPCS)
      Mean: 0.925 ± 0.065
      Range: [0.800, 1.000]

   Clinical Realism Score (CRS)
      Mean: 0.792 ± 0.098
      Range: [0.625, 0.900]

   🎯 OVERALL AVERAGE: 0.858


📈 PERFORMANCE BY DISEASE:

   Top 3 Best Performing:
      1. Simple Cavities: 0.912
         (SFS: 0.950, RPCS: 0.950, CRS: 0.837)
      2. Reversible Pulpitis: 0.888
         (SFS: 0.900, RPCS: 1.000, CRS: 0.762)
      3. Periodontal Abscess: 0.862
         (SFS: 0.812, RPCS: 0.975, CRS: 0.800)

   Bottom 3 Need Improvement:
      1. Pulp Necrosis: 0.791
         (SFS: 0.750, RPCS: 0.850, CRS: 0.775)
      2. Acute Apical Periodontitis: 0.806
         (SFS: 0.800, RPCS: 0.900, CRS: 0.718)
      3. Pericoronitis: 0.823
         (SFS: 0.833, RPCS: 0.900, CRS: 0.737)

✅ Results saved to: test_results_20241202_143052.json
======================================================================
"""


# ============================================================================
# RECOMMENDED WORKFLOW
# ============================================================================

print("\n" + "="*70)
print("RECOMMENDED TESTING WORKFLOW")
print("="*70)
print("""
1. Start with QUICK TEST (5 conversations)
   - Get a feel for how your model performs
   - See if there are obvious issues

2. If Quick Test looks good, run FULL TEST SUITE
   - Tests all diseases systematically
   - Gives you comprehensive metrics
   - Saves detailed JSON for analysis

3. Look at the results:
   - Which diseases perform well?
   - Which need improvement?
   - Is SFS low? → Model isn't using correct symptoms
   - Is RPCS low? → Model breaks character
   - Is CRS low? → Responses aren't realistic

4. Deep dive into problem areas:
   - Use OPTION 3 to test specific diseases
   - Check the detailed violation examples
   - Review missing/hallucinated symptoms

5. Iterate:
   - Adjust your training data
   - Modify system prompts
   - Re-run tests to measure improvement
""")

print("="*70)
print("Ready to test! Uncomment one of the options above to start.")
print("="*70)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


OPTION 1: QUICK TEST
Runs 5 random conversations with your LLM and evaluates them

🚀 Running Quick Test...
   5 random conversations


Testing: Simple Cavities
Expected symptoms: 6
Conversation length: medium (13 questions)

[Q1] Doctor: Welcome to the clinic. What can I help you with today?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[A1] Patient: I've been having some issues with my teeth lately. When I brush my teeth, I feel a bit uncomfortable.

[Q2] Doctor: Where exactly in your mouth is the pain located? Can you point to the specific tooth or area?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

[A8] Patient: No, my mouth opens normally. I can open it wide without any issues.

[Q9] Doctor: Have you noticed any bad breath or odor coming from your mouth?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[A9] Patient: No, my breath feels normal. I don't notice anything unusual.

[Q10] Doctor: Have you noticed any changes in the color of your teeth or gums?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[A10] Patient: My gums look a bit swollen, but the color seems normal.

[Q11] Doctor: Is there anything you'd like to ask me about your symptoms?
[A11] Patient: No, I'm just trying to figure out what's going on.


PATIENT SIMULATION EVALUATION REPORT

📊 OVERALL SCORES:
   Symptom Fidelity Score (SFS):      0.833
   Role-Playing Consistency (RPCS):   1.000
   Clinical Realism Score (CRS):      0.875
   Average Score:                      0.903

🔍 DETAILED BREAKDOWN:

   Symptom Fidelity:
   - Expected symptoms: 6
   - Correctly mentioned: 5
   - Missing: salty-tasting fluid

   Role-Playing Consistency:
   - Total responses: 11
   - Violation rate: 0.0%

   Clinical Realism:
   - Symptom description: 0.50
   - Vocabulary: 1.00
   - Pain description: 1.00
   - Consistency: 1.00



QUICK TEST SUMMARY
Average SFS:  0.938
Average RPCS: 0.960
Average CRS:  0.925
Overall:      0.941


OPTION 2: FULL TEST SUITE
Tests all 8 diseases with 2-3 conversations each
Generates comprehensive report + s